# GEX implied book — gamma exposure vs RV/range

**Paper:** Gamma → $/pt implied LOB (PDF pp. 4–5) · **NOTES:** [`NOTES.md`](NOTES.md) · **EXP:** [`EXP_REPORT.md`](EXP_REPORT.md)  
**Artifacts:** `out/gex_implied_book/` · Label: `BS_GEX_trade_flow_DDOI`

$$\mathrm{GEX} = \sum_i \mathrm{DDOI}_i \cdot \gamma_i \cdot \$\mathrm{mult}$$

DDOI = **PROXY_trade_flow_DDOI**. IV = Deribit warehouse options. Underlying S = Deribit ETH-PERPETUAL mark. HL TOB for RV/range join — **Kill** as α.


In [ ]:
from pathlib import Path
import json, os, sys
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("module://matplotlib_inline.backend_inline")
import matplotlib.pyplot as plt
from IPython.display import Image, display, Markdown

BOOK = Path(os.environ.get("ARES_MICROSTRUCTURE") or (Path.home() / "srv" / "ares-microstructure")) / "research" / "books" / "squeeze_metrics"
OUT = BOOK / "out"
SCRIPTS = BOOK / "scripts"
ROOT = BOOK.parents[2]
sys.path.insert(0, str(SCRIPTS))
sys.path.insert(0, str(ROOT))
from _nb_common import (
    load_json, gex_day_table, safe_corr, partial_corr_vs_rv, lead_lag_day, SIGNAL_BOARD,
)
from certified_panel import load_certified, primary_days, spot_l2_days, gex_panel_days, DDOI_LABEL, GEX_PROXY_LABEL
from research.lib import squeeze  # noqa: E402

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})

def show_pngs(fig_dir: Path, pattern="*.png"):
    if not fig_dir.is_dir():
        print("no figs dir", fig_dir)
        return
    for p in sorted(fig_dir.glob(pattern)):
        display(Markdown(f"**{p.name}** — `{p.relative_to(BOOK)}`"))
        display(Image(filename=str(p)))

def print_board(rows=None):
    rows = rows or SIGNAL_BOARD
    print(f"{'id':34s} {'gate':6s} note")
    print("-" * 88)
    for i, g, n in rows:
        print(f"{i:34s} {g:6s} {n}")

CERT = load_certified()
PRIMARY = primary_days(CERT)
SPOT = spot_l2_days(CERT)
GEX_DAYS = gex_panel_days(CERT)
print("BOOK", BOOK)
print("certified primary n=", len(PRIMARY), PRIMARY)
print("panel_gex_options n=", len(GEX_DAYS), GEX_DAYS)
print("spot_l2 subpanel n=", len(SPOT), SPOT)
print("DDOI label", DDOI_LABEL, "| GEX proxy", GEX_PROXY_LABEL)
print("trade_synth QUARANTINED — never primary; warehouse OI futures-only unused")


In [ ]:

summary = load_json(OUT / "gex_implied_book" / "summary.json")
panel = load_json(OUT / "gex_implied_book" / "panel_rows.json") or (summary or {}).get("rows") or []
tbl = gex_day_table(panel, CERT)
print(json.dumps({k: (summary or {}).get(k) for k in (
    "n_ok","days_ok","gex_mean","vex_mean","gex_plus_mean",
    "corr_gex_hl_rv","corr_gex_hl_range","corr_gexplus_hl_range",
    "ddoi_modes","decision","promote","kill_tob_cross_alpha","title",
)}, indent=2, default=str))
display(tbl.round(5) if len(tbl) else tbl)


## Existing figs


In [ ]:
show_pngs(OUT / "gex_implied_book" / "figs")


## GEX vs HL RV / range + stress split


In [ ]:

ok = tbl[tbl["ok"] == True].copy() if len(tbl) else tbl
fig, axes = plt.subplots(1, 3, figsize=(12.8, 4.0))
if len(ok):
    axes[0].bar(ok["day"], ok["gex"], color="#3d5a80")
    axes[0].axhline(0, color="k", lw=0.8)
    axes[0].tick_params(axis="x", rotation=70)
    axes[0].set_title("Day GEX")
    axes[1].scatter(ok["gex"], ok["hl_rv"], s=50, c="#ee6c4d")
    for _, r in ok.iterrows():
        axes[1].annotate(str(r["day"])[5:], (r["gex"], r["hl_rv"]), fontsize=7, alpha=0.8)
    axes[1].set_xlabel("GEX"); axes[1].set_ylabel("HL RV")
    axes[1].set_title(f"corr={safe_corr(ok['gex'], ok['hl_rv'])}")
    med = float(ok["hl_range"].median()) if len(ok) else np.nan
    ok["stress"] = np.where(ok["hl_range"] >= med, "stress", "calm")
    for lab, c in (("calm", "#98c1d9"), ("stress", "#ee6c4d")):
        g = ok[ok["stress"] == lab]
        axes[2].scatter(g["gex"], g["hl_range"], s=55, c=c, label=f"{lab} n={len(g)}")
    axes[2].set_xlabel("GEX"); axes[2].set_ylabel("HL range")
    axes[2].set_title(f"stress split (med range={med:.4f})")
    axes[2].legend(fontsize=8)
else:
    for ax in axes:
        ax.text(0.5, 0.5, "run exp_gex_panel.py", ha="center", transform=ax.transAxes)
plt.tight_layout(); plt.show()
if len(ok):
    print("corr GEX~RV", safe_corr(ok["gex"], ok["hl_rv"]), "GEX~range", safe_corr(ok["gex"], ok["hl_range"]))
    print(ok.groupby("stress")[["gex","hl_rv","hl_range"]].mean().round(6))


## Venue / DDOI honesty


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(12.0, 3.8))
if len(ok):
    x = np.arange(len(ok)); w = 0.35
    axes[0].bar(x - w/2, ok["opt_iv_n"].fillna(0), w, label="opt IV n", color="#3d5a80")
    axes[0].bar(x + w/2, ok["opt_trade_n"].fillna(0), w, label="opt trades", color="#ee6c4d")
    axes[0].set_xticks(x); axes[0].set_xticklabels(ok["day"], rotation=70, fontsize=8)
    axes[0].set_title("Options coverage"); axes[0].legend(fontsize=8)
    axes[1].bar(ok["day"], ok["hl_tob_n"].fillna(0), color="#98c1d9")
    axes[1].tick_params(axis="x", rotation=70)
    axes[1].set_title("HL TOB n (real quotes)")
else:
    for ax in axes:
        ax.text(0.5, 0.5, "no panel", ha="center", transform=ax.transAxes)
plt.tight_layout(); plt.show()
if len(ok):
    print(ok[["day","ddoi_mode","kraken","opt_iv_n","opt_trade_n","hl_tob_n"]].to_string(index=False))
print("Warehouse OI = futures-only → unused. trade_synth QUARANTINED.")


## Lib smoke (synthetic)


In [ ]:

# toy chain — formulas compute without warehouse
flags = np.array(["c","p","c","p"])
S, K = 2500.0, np.array([2400., 2400., 2600., 2600.])
T = np.full(4, 7/365); iv = np.full(4, 0.55)
ddoi = squeeze.ddoi_from_trade_flow(np.array([10., -8., 5., -12.]), np.array([1, -1, 1, -1]))
pack = squeeze.chain_exposures(
    flags=flags, strikes=K, ttm_years=T, iv=iv, ddoi=ddoi, spot=S, r=0.0,
)
print("toy GEX/VEX/GEX+", pack["gex"], pack["vex"], pack["gex_plus"], "label", squeeze.LABEL_TRADE_DDOI)


## Signal board


In [ ]:

print_board([
    ("risk.gex_exposure", "Hold", f"n_ok={(summary or {}).get('n_ok')}; {GEX_PROXY_LABEL}"),
    ("liq.gex_rv_link", "Hold", f"corr_rv={(summary or {}).get('corr_gex_hl_rv')} corr_range={(summary or {}).get('corr_gex_hl_range')}"),
    ("alpha.tob_cross_arb", "Kill", "co-move ≠ arb"),
    ("data.trade_synth", "Kill", "quarantined"),
])
